# Story 2 — The "Real" Delay Calculator
**Goal:** Calculate the difference between the estimated and actual delivery date, then classify every order as *On Time*, *Late*, or *Super Late*.

**Input:** `../data/01_master.parquet`  
**Output:** `../data/02_delivered.parquet`

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

COLOR_MAP  = {'On Time': '#2ecc71', 'Late': '#f39c12', 'Super Late': '#e74c3c'}
ORDER_CATS = ['On Time', 'Late', 'Super Late']
print('Libraries loaded.')

## Step 1 — Load master data

In [ ]:
master = pd.read_parquet('../data/01_master.parquet')
print(f'Loaded: {master.shape[0]:,} rows')
print(f'Columns: {list(master.columns)}')

## Step 2 — Parse datetime columns

In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
    master[col] = pd.to_datetime(master[col])

print('Date columns parsed successfully.')
master[date_cols].dtypes

## Step 3 — Handle missing values
Orders with status `canceled` or `unavailable`, or with no actual delivery date, are **excluded** from delay analysis and flagged separately.

In [ ]:
print('Order status breakdown:')
print(master['order_status'].value_counts().to_string())

excluded = master[
    master['order_status'].isin(['canceled', 'unavailable']) |
    master['order_delivered_customer_date'].isna()
].copy()

delivered = master[
    (~master['order_status'].isin(['canceled', 'unavailable'])) &
    master['order_delivered_customer_date'].notna()
].copy()

print(f'\nTotal:    {len(master):,}')
print(f'Excluded: {len(excluded):,}  (canceled, unavailable, or never delivered)')
print(f'Kept:     {len(delivered):,}  (delivered orders — used for all analysis)')

## Step 4 — Calculate `delivery_delay_days`
```
delivery_delay_days = actual_delivery_date − estimated_delivery_date
```
- **Negative** = arrived early (good)  
- **Zero** = arrived exactly on the estimated date  
- **Positive** = arrived late (bad — we are lying to customers)

In [ ]:
delivered['delivery_delay_days'] = (
    delivered['order_delivered_customer_date'] -
    delivered['order_estimated_delivery_date']
).dt.days

print('Delay statistics (days):')
print(delivered['delivery_delay_days'].describe().round(1).to_string())

## Step 5 — Classify orders
| Status | Condition |
|--------|----------|
| On Time | delay ≤ 0 days |
| Late | 1–5 days late |
| Super Late | > 5 days late |

In [ ]:
def classify_delivery(days):
    if days <= 0:   return 'On Time'
    elif days <= 5: return 'Late'
    else:           return 'Super Late'

delivered['delivery_status'] = delivered['delivery_delay_days'].apply(classify_delivery)

counts = delivered['delivery_status'].value_counts()
pcts   = delivered['delivery_status'].value_counts(normalize=True) * 100

print('Delivery Status Summary:')
print(f'{"Status":<12}  {"Orders":>8}  {"Pct":>6}')
print('-' * 32)
for s in ORDER_CATS:
    print(f'{s:<12}  {counts[s]:>8,}  {pcts[s]:>5.1f}%')

## Step 6 — Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
vals = [counts[s] for s in ORDER_CATS]
bars = axes[0].bar(ORDER_CATS, vals, color=[COLOR_MAP[s] for s in ORDER_CATS], edgecolor='white', linewidth=1.5)
axes[0].set_title('Order Count by Delivery Status', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
p = [pcts[s] for s in ORDER_CATS]
axes[1].pie(p, labels=[f'{s}\n{v:.1f}%' for s, v in zip(ORDER_CATS, p)],
            colors=[COLOR_MAP[s] for s in ORDER_CATS],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Delivery Status Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../exports/fig_delivery_status.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig = px.histogram(
    delivered, x='delivery_delay_days', nbins=80,
    color='delivery_status', color_discrete_map=COLOR_MAP,
    category_orders={'delivery_status': ORDER_CATS},
    title='Distribution of Delivery Delay (days)',
    labels={'delivery_delay_days': 'Days (negative = early, positive = late)'},
    opacity=0.85,
)
fig.add_vline(x=0, line_dash='dash', line_color='black', annotation_text='Estimated date')
fig.update_layout(bargap=0.05, legend_title='Status')
fig.show()

## Step 7 — Save for next notebooks

In [ ]:
delivered.to_parquet('../data/02_delivered.parquet', index=False)
print('Saved → ../data/02_delivered.parquet')
print('\nNext: run 03_geographic_heatmap.ipynb')